# Lab 4: Motors and Open Loop Control

Run each cell top-to-bottom. Each section corresponds to a lab task for documentation.

**PWM range: 0-255 (Artemis Nano default)**

In [56]:
!pip install bleak colorama numpy matplotlib bleach PyYAML  --upgrade


In [57]:
%load_ext autoreload
%autoreload 2

from ble import get_ble_controller
from base_ble import LOG
from cmd_types import CMD
import time
import numpy as np
import matplotlib.pyplot as plt

LOG.propagate = False

PWM_MAX = 255  # Artemis Nano analogWrite() range: 0-255


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [58]:
ble = get_ble_controller()
ble.connect()


2026-03-10 02:29:57,078 | INFO     |: Looking for Artemis Nano Peripheral Device: c0:81:95:21:a2:64
2026-03-10 02:29:57,079 | INFO     |: Scanning for device with address: c0:81:95:21:a2:64, service UUID: 1785129f-3b3a-4cf5-a01f-03668e8b12e9
2026-03-10 02:30:07,129 | INFO     |: Found 1 device(s) advertising service 1785129f-3b3a-4cf5-a01f-03668e8b12e9
2026-03-10 02:30:07,130 | INFO     |: Selecting device: AE68F85F-67D4-4FE1-04EC-9F24C2B315FC (name: Artemis BLE)
2026-03-10 02:30:08,027 | INFO     |: Connected to c0:81:95:21:a2:64


## Notification Handler + Safety Timeout

In [59]:
transfer_done = False
last_message = ""

def handle_data(uuid, message):
    global transfer_done, last_message
    msg = message.decode()
    last_message = msg
    print(msg)
    if msg == "END":
        transfer_done = True

try:
    ble.stop_notify(ble.uuid["RX_STRING"])
except Exception:
    pass

ble.start_notify(ble.uuid["RX_STRING"], handle_data)

# Safety timeout: motors auto-stop after 3 seconds
ble.send_command(CMD.MOTOR_TIMEOUT, "3000")
print("Ready -- 3s safety timeout active")


Ready -- 3s safety timeout active


---
## Task 4: Single Motor Spin Test (Wheels Elevated)
Put the car on its side. Tests each motor forward and reverse.

**Pin mapping:** Motor 1 (left) = A0/A1, Motor 2 (right) = A2/A3

In [60]:
# Full throttle test -- 255 is max PWM
# Motor 1 forward full
print("=== Motor 1 Full Forward ===")
ble.send_command(CMD.MOTOR_CMD, "255|0")
time.sleep(3)
ble.send_command(CMD.MOTOR_STOP, "")
time.sleep(1)

# Motor 1 reverse full
print("=== Motor 1 Full Reverse ===")
ble.send_command(CMD.MOTOR_CMD, "-255|0")
time.sleep(3)
ble.send_command(CMD.MOTOR_STOP, "")
time.sleep(1)

# Motor 2 forward full
print("=== Motor 2 Full Forward ===")
ble.send_command(CMD.MOTOR_CMD, "0|255")
time.sleep(3)
ble.send_command(CMD.MOTOR_STOP, "")
time.sleep(1)

# Motor 2 reverse full
print("=== Motor 2 Full Reverse ===")
ble.send_command(CMD.MOTOR_CMD, "0|-255")
time.sleep(3)
ble.send_command(CMD.MOTOR_STOP, "")
print("Done -- check serial for 'Motors: L=255 R=0 PWM_MAX=255' etc.")


=== Motor 1 Full Forward ===
=== Motor 1 Full Reverse ===
=== Motor 2 Full Forward ===
=== Motor 2 Full Reverse ===
Done -- check serial for 'Motors: L=255 R=0 PWM_MAX=255' etc.


---
## Task 8: Minimum PWM Threshold
Place the robot on the floor. Sweeps PWM from low to high to find the lowest value that moves the robot.

Watch carefully and note which PWM value first causes motion.

In [61]:
# Forward sweep: find minimum PWM to start moving from rest
for pwm in range(20, 180, 10):
    print(f"PWM = {pwm}")
    ble.send_command(CMD.MOTOR_CMD, f"{pwm}|{pwm}")
    time.sleep(1.5)
    ble.send_command(CMD.MOTOR_STOP, "")
    time.sleep(1.0)

print("Forward sweep done")


PWM = 20
PWM = 30
PWM = 40
PWM = 50
PWM = 60
PWM = 70
PWM = 80
PWM = 90
PWM = 100
PWM = 110
PWM = 120
PWM = 130


BleakError: disconnected

In [ ]:
# On-axis turn sweep: find minimum PWM for rotation
for pwm in range(20, 180, 10):
    print(f"Turn PWM = {pwm}")
    ble.send_command(CMD.MOTOR_CMD, f"{pwm}|{-pwm}")
    time.sleep(1.5)
    ble.send_command(CMD.MOTOR_STOP, "")
    time.sleep(1.0)

print("Turn sweep done")


Turn PWM = 20
Turn PWM = 30
Turn PWM = 40
Turn PWM = 50
Turn PWM = 60
Turn PWM = 70
Turn PWM = 80
Turn PWM = 90
Turn PWM = 100
Turn PWM = 110
Turn PWM = 120
Turn PWM = 130
Turn PWM = 140
Turn PWM = 150
Turn PWM = 160
Turn PWM = 170
Turn sweep done


---
## Task 9: Straight-Line Calibration
Adjust `motor2_cal` so the robot drives straight for 2+ meters.
- `> 1.0` if motor 2 is slower (robot veers right)
- `< 1.0` if motor 2 is faster (robot veers left)

Start on a tape line, drive forward, check if it still overlaps at the end.

In [ ]:
# Set calibration factor -- tune this value
CAL_FACTOR = 1.0  # adjust empirically
ble.send_command(CMD.MOTOR_CAL, str(CAL_FACTOR))

# Drive straight for calibration test (record video!)
DRIVE_PWM = 150
ble.send_command(CMD.MOTOR_TIMEOUT, "5000")
time.sleep(0.1)
ble.send_command(CMD.MOTOR_CMD, f"{DRIVE_PWM}|{DRIVE_PWM}")
time.sleep(4)
ble.send_command(CMD.MOTOR_STOP, "")


CAL:1.000


---
## Task 10: Open-Loop Autonomous Demo
Untethered sequence: forward, turn, forward, turn, forward.

Record video for write-up!

In [ ]:
DRIVE = 150   # cruising PWM
TURN = 120    # turning PWM
ble.send_command(CMD.MOTOR_TIMEOUT, "15000")

# Forward
print("Forward")
ble.send_command(CMD.MOTOR_CMD, f"{DRIVE}|{DRIVE}")
time.sleep(2)

# Right turn (spin in place)
print("Right turn")
ble.send_command(CMD.MOTOR_CMD, f"{TURN}|{-TURN}")
time.sleep(0.5)

# Forward
print("Forward")
ble.send_command(CMD.MOTOR_CMD, f"{DRIVE}|{DRIVE}")
time.sleep(2)

# Left turn
print("Left turn")
ble.send_command(CMD.MOTOR_CMD, f"{-TURN}|{TURN}")
time.sleep(0.5)

# Forward
print("Forward")
ble.send_command(CMD.MOTOR_CMD, f"{DRIVE}|{DRIVE}")
time.sleep(2)

ble.send_command(CMD.MOTOR_STOP, "")
print("Sequence complete")


Forward
Right turn
Forward
Left turn
Forward
Sequence complete


---
## Emergency Stop + Disconnect

In [ ]:
ble.send_command(CMD.MOTOR_STOP, "")
ble.disconnect()
print("Stopped and disconnected")


Stopped and disconnected
